In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
methods_names = {
    "dist_linguistic_confidence": "Dist. Ling. Conf.",
    "dist_semantic_uncertainty": "Dist. Semantic Unc.",
    "dist_lnll": "Dist. Token Prob"
}

dataset_size = {
    "mmlu": 116000,
    "squadv2": 130000,
    "truthful_qa": 817
}

dataset_map = {
    "mmlu": "MMLU",
    "squadv2": "SQuAD2.0",
    "truthful_qa": "TruthfulQA"
}

model_name_map = {
    "Llama-3.1-8B-Instruct": "Llama-3.1-8B-Inst.",
    "Meta-Llama-3-8B-Instruct": "Llama-3-8B-Inst.",
    "Qwen2.5-7B-Instruct": "Qwen2.5-7B-Inst.",
    "Qwen3-8B": "Qwen3-8B-Inst.",
    "Mistral-7B-Instruct-v0.3": "Mistral-7B-Inst.",
    "gpt-oss-20b": "GPT-OSS-20B",
}

# Cross domain data

In [3]:
prompt_type = "direct_qa"

In [4]:
results_dir = f"/hdd/ivny/{prompt_type}_cross_domain_calibration"

# list all leaf nodes in results_dir
leaf_dirs = []
for root, dirs, files in os.walk(results_dir):
    if not dirs:  # if there are no subdirectories, it's a leaf node
        leaf_dirs.append(root)

all_records = []
for leaf_dir in leaf_dirs:
    _, _, _, _, dataset_name, _, model_name = leaf_dir.split("/")
    training_set, test_set = dataset_name.split("--")
    if os.path.exists(os.path.join(leaf_dir, "calibration_performance.csv")):
        csv_data = pd.read_csv(os.path.join(leaf_dir, "calibration_performance.csv"), index_col=0)
        csv_records = csv_data.transpose().to_dict("records")

        for record in csv_records:
            record["model"] = model_name_map.get(model_name, model_name)
            record["training_set"] = dataset_map.get(training_set, training_set)
            record["test_set"] = dataset_map.get(test_set, test_set)
            record["dataset_size"] = dataset_size.get(test_set, None)

        all_records.extend(csv_records)
    else:
        print(model_name, dataset_name, "missing calibration_performance.csv")

In [5]:
df = pd.DataFrame(all_records).drop(columns=["model", "dataset_size"])
dataset_weighted_average_mean = df.groupby(["training_set", "test_set"]).mean().reset_index()

In [6]:
dataset_weighted_average_mean

,training_set,test_set,original_lc_generalised_ECE,original_lc_faithfulness_divergence,original_lc_ece_mean,original_lc_dAUROC,original_lc_auroc_mean,original_tp_generalised_ECE,original_tp_faithfulness_divergence,original_tp_ece_mean,...,calibrated_tp_rewritten_lc_generalised_ECE,calibrated_tp_rewritten_lc_faithfulness_divergence,calibrated_tp_rewritten_lc_ece_mean,calibrated_tp_rewritten_lc_dAUROC,calibrated_tp_rewritten_lc_auroc_mean,calibrated_su_rewritten_lc_generalised_ECE,calibrated_su_rewritten_lc_faithfulness_divergence,calibrated_su_rewritten_lc_ece_mean,calibrated_su_rewritten_lc_dAUROC,calibrated_su_rewritten_lc_auroc_mean
0,MMLU,SQuAD2.0,0.353680,1.984678,0.327808,0.510150,0.506597,0.300746,48.923606,0.295621,...,0.208730,0.822788,0.169957,0.563875,0.582900,0.233518,1.105980,0.228826,0.601500,0.618076
1,MMLU,TruthfulQA,0.407558,1.979075,0.401492,0.578775,0.616875,0.239099,13.643382,0.233666,...,0.256223,0.774945,0.218388,0.560550,0.604918,0.261444,0.976019,0.254936,0.642400,0.671718
2,SQuAD2.0,MMLU,0.186957,0.901116,0.132444,0.529600,0.542357,0.267733,141.401255,0.251930,...,0.142622,0.604937,0.113046,0.530575,0.544390,0.136376,0.485378,0.099081,0.609625,0.643760
3,SQuAD2.0,TruthfulQA,0.407416,1.979075,0.401492,0.583725,0.616875,0.238998,13.643382,0.233666,...,0.155937,0.527403,0.121323,0.591275,0.631424,0.198773,0.617740,0.181789,0.638425,0.673258
4,TruthfulQA,MMLU,0.187077,0.901116,0.132444,0.532375,0.542357,0.267723,141.401255,0.251930,...,0.143854,0.572921,0.106514,0.519275,0.528408,0.123641,0.453354,0.091960,0.593850,0.635959
5,TruthfulQA,SQuAD2.0,0.353686,1.984678,0.327808,0.507250,0.506597,0.300764,48.923606,0.295621,...,0.183665,0.683333,0.171277,0.543025,0.566713,0.145742,0.633787,0.138401,0.575725,0.603194


In [7]:
dataset_weighted_average_mean.columns.tolist()

['training_set',
 'test_set',
 'original_lc_generalised_ECE',
 'original_lc_faithfulness_divergence',
 'original_lc_ece_mean',
 'original_lc_dAUROC',
 'original_lc_auroc_mean',
 'original_tp_generalised_ECE',
 'original_tp_faithfulness_divergence',
 'original_tp_ece_mean',
 'original_tp_dAUROC',
 'original_tp_auroc_mean',
 'original_su_generalised_ECE',
 'original_su_faithfulness_divergence',
 'original_su_ece_mean',
 'original_su_dAUROC',
 'original_su_auroc_mean',
 'calibrated_lc_generalised_ECE',
 'calibrated_lc_faithfulness_divergence',
 'calibrated_lc_ece_mean',
 'calibrated_lc_dAUROC',
 'calibrated_lc_auroc_mean',
 'calibrated_tp_generalised_ECE',
 'calibrated_tp_faithfulness_divergence',
 'calibrated_tp_ece_mean',
 'calibrated_tp_dAUROC',
 'calibrated_tp_auroc_mean',
 'calibrated_su_generalised_ECE',
 'calibrated_su_faithfulness_divergence',
 'calibrated_su_ece_mean',
 'calibrated_su_dAUROC',
 'calibrated_su_auroc_mean',
 'calibrated_lc_rewritten_lc_generalised_ECE',
 'calibrate

In [8]:
pct = False 

In [9]:
fd_improvement_df = dataset_weighted_average_mean[["training_set", "test_set"]].copy()


fd_improvement_df["Linguistic Confidence"] = (dataset_weighted_average_mean["calibrated_lc_rewritten_lc_faithfulness_divergence"] - dataset_weighted_average_mean["original_lc_faithfulness_divergence"]) 
fd_improvement_df["Token Probability"] = (dataset_weighted_average_mean["calibrated_tp_rewritten_lc_faithfulness_divergence"] - dataset_weighted_average_mean["original_lc_faithfulness_divergence"]) 
fd_improvement_df["Semantic Uncertainty"] = (dataset_weighted_average_mean["calibrated_su_rewritten_lc_faithfulness_divergence"] - dataset_weighted_average_mean["original_lc_faithfulness_divergence"]) 

if pct:
    fd_improvement_df["Linguistic Confidence"] = fd_improvement_df["Linguistic Confidence"] / dataset_weighted_average_mean["original_lc_faithfulness_divergence"]
    fd_improvement_df["Token Probability"] = fd_improvement_df["Token Probability"] / dataset_weighted_average_mean["original_lc_faithfulness_divergence"]
    fd_improvement_df["Semantic Uncertainty"] = fd_improvement_df["Semantic Uncertainty"] / dataset_weighted_average_mean["original_lc_faithfulness_divergence"]

fd_improvement_df.to_dict("records")

[{'training_set': 'MMLU',
  'test_set': 'SQuAD2.0',
  'Linguistic Confidence': -0.8792312847259112,
  'Token Probability': -1.1618895615795726,
  'Semantic Uncertainty': -0.8786972106993292},
 {'training_set': 'MMLU',
  'test_set': 'TruthfulQA',
  'Linguistic Confidence': -0.8043972136173727,
  'Token Probability': -1.204129530362997,
  'Semantic Uncertainty': -1.0030552894795999},
 {'training_set': 'SQuAD2.0',
  'test_set': 'MMLU',
  'Linguistic Confidence': -0.4021494691540988,
  'Token Probability': -0.2961795985910919,
  'Semantic Uncertainty': -0.4157384330200574},
 {'training_set': 'SQuAD2.0',
  'test_set': 'TruthfulQA',
  'Linguistic Confidence': -1.2447881060506591,
  'Token Probability': -1.4516711667986841,
  'Semantic Uncertainty': -1.3613347721832088},
 {'training_set': 'TruthfulQA',
  'test_set': 'MMLU',
  'Linguistic Confidence': -0.34150915413911,
  'Token Probability': -0.32819558506086266,
  'Semantic Uncertainty': -0.4477627129529855},
 {'training_set': 'TruthfulQA',


In [10]:
ece_improvement_df  = dataset_weighted_average_mean[["training_set", "test_set"]].copy()

ece_improvement_df["Linguistic Confidence"] = (dataset_weighted_average_mean["calibrated_lc_rewritten_lc_generalised_ECE"] - dataset_weighted_average_mean["original_lc_generalised_ECE"]) 
ece_improvement_df["Token Probability"] = (dataset_weighted_average_mean["calibrated_tp_rewritten_lc_generalised_ECE"] - dataset_weighted_average_mean["original_lc_generalised_ECE"]) 
ece_improvement_df["Semantic Uncertainty"] = (dataset_weighted_average_mean["calibrated_su_rewritten_lc_generalised_ECE"] - dataset_weighted_average_mean["original_lc_generalised_ECE"]) 

if pct:
    ece_improvement_df["Linguistic Confidence"] = ece_improvement_df["Linguistic Confidence"] / dataset_weighted_average_mean["original_lc_generalised_ECE"]
    ece_improvement_df["Token Probability"] = ece_improvement_df["Token Probability"] / dataset_weighted_average_mean["original_lc_generalised_ECE"]
    ece_improvement_df["Semantic Uncertainty"] = ece_improvement_df["Semantic Uncertainty"] / dataset_weighted_average_mean["original_lc_generalised_ECE"]

ece_improvement_df.to_dict("records")

[{'training_set': 'MMLU',
  'test_set': 'SQuAD2.0',
  'Linguistic Confidence': -0.08336658957415977,
  'Token Probability': -0.14494971160813533,
  'Semantic Uncertainty': -0.12016201049659833},
 {'training_set': 'MMLU',
  'test_set': 'TruthfulQA',
  'Linguistic Confidence': -0.05835608557364219,
  'Token Probability': -0.1513345846817859,
  'Semantic Uncertainty': -0.14611404438347375},
 {'training_set': 'SQuAD2.0',
  'test_set': 'MMLU',
  'Linguistic Confidence': -0.03908998273946354,
  'Token Probability': -0.044334677888776874,
  'Semantic Uncertainty': -0.05058052393607235},
 {'training_set': 'SQuAD2.0',
  'test_set': 'TruthfulQA',
  'Linguistic Confidence': -0.14248405226698735,
  'Token Probability': -0.25147943014037066,
  'Semantic Uncertainty': -0.2086437071876196},
 {'training_set': 'TruthfulQA',
  'test_set': 'MMLU',
  'Linguistic Confidence': 0.02304626065517107,
  'Token Probability': -0.043223139371820374,
  'Semantic Uncertainty': -0.06343676651239466},
 {'training_set'

# In domain diagonal data

In [11]:
results_dir = f"/hdd/ivny/{prompt_type}_in_domain_calibration"

# list all leaf nodes in results_dir
leaf_dirs = []
for root, dirs, files in os.walk(results_dir):
    if not dirs:  # if there are no subdirectories, it's a leaf node
        leaf_dirs.append(root)

all_records = []
for leaf_dir in leaf_dirs:
    _, _, _, _, dataset_name, _, model_name = leaf_dir.split("/")
    if os.path.exists(os.path.join(leaf_dir, "calibration_performance.csv")):
        csv_data = pd.read_csv(os.path.join(leaf_dir, "calibration_performance.csv"), index_col=0)
        csv_records = csv_data.transpose().to_dict("records")

        for record in csv_records:
            record["dataset"] = dataset_map.get(dataset_name, dataset_name)
            record["model"] = model_name_map.get(model_name, model_name)
            record["dataset_size"] = dataset_size.get(dataset_name, None)

        all_records.extend(csv_records)
    else:
        print(model_name, dataset_name, "missing calibration_performance.csv")

In [12]:
full_df = pd.DataFrame(all_records)
full_df = full_df.drop(columns=["model", "dataset_size"]).groupby("dataset").mean().reset_index()

in_domain_ece = []
in_domain_fd = []

for _, row in full_df.iterrows():
    if pct:
        ece = {
            'training_set': row['dataset'],
            'test_set': row['dataset'],
            'Linguistic Confidence': (row["calibrated_lc_rewritten_lc_generalised_ECE"] - row['original_lc_generalised_ECE']) / row['original_lc_generalised_ECE'],
            'Token Probability': (row["calibrated_tp_rewritten_lc_generalised_ECE"] - row['original_lc_generalised_ECE']) / row['original_lc_generalised_ECE'],
            'Semantic Uncertainty': (row["calibrated_su_rewritten_lc_generalised_ECE"] - row['original_lc_generalised_ECE']) / row['original_lc_generalised_ECE']
        }

        fd = {
            'training_set': row['dataset'],
            'test_set': row['dataset'],
            'Linguistic Confidence': (row["calibrated_lc_rewritten_lc_faithfulness_divergence"] - row['original_lc_faithfulness_divergence']) / row['original_lc_faithfulness_divergence'],
            'Token Probability': (row["calibrated_tp_rewritten_lc_faithfulness_divergence"] - row['original_lc_faithfulness_divergence']) / row['original_lc_faithfulness_divergence'],
            'Semantic Uncertainty': (row["calibrated_su_rewritten_lc_faithfulness_divergence"] - row['original_lc_faithfulness_divergence']) / row['original_lc_faithfulness_divergence']
        }
    else:
        ece = {
            'training_set': row['dataset'],
            'test_set': row['dataset'],
            'Linguistic Confidence': row["calibrated_lc_rewritten_lc_generalised_ECE"] - row['original_lc_generalised_ECE'],
            'Token Probability': row["calibrated_tp_rewritten_lc_generalised_ECE"] - row['original_lc_generalised_ECE'],
            'Semantic Uncertainty': row["calibrated_su_rewritten_lc_generalised_ECE"] - row['original_lc_generalised_ECE']
        }

        fd = {
            'training_set': row['dataset'],
            'test_set': row['dataset'],
            'Linguistic Confidence': row["calibrated_lc_rewritten_lc_faithfulness_divergence"] - row['original_lc_faithfulness_divergence'],
            'Token Probability': row["calibrated_tp_rewritten_lc_faithfulness_divergence"] - row['original_lc_faithfulness_divergence'],
            'Semantic Uncertainty': row["calibrated_su_rewritten_lc_faithfulness_divergence"] - row['original_lc_faithfulness_divergence']
        }
    in_domain_ece.append(ece)
    in_domain_fd.append(fd)


In [13]:
in_domain_ece_df = pd.DataFrame(in_domain_ece)
in_domain_fd_df = pd.DataFrame(in_domain_fd)

# Format latex table

In [14]:
def generate_latex_table(faithfulness_data, ece_data, pct):
    def fmt(value, pct=False):
        if value is None:
            return r"-"
        if pct:
            pct = value * 100
            color = "green!70!black" if pct < 0 else "red!70!black"
            abs_pct = abs(pct)
            return rf"\textcolor{{{color}}}{{$\Delta${abs_pct:.2f}\%}}"
        else:
            color = "green!70!black" if value < 0 else "red!70!black"
            abs_value = abs(value)
            return rf"\textcolor{{{color}}}{{$\Delta${abs_value:.4f}}}"

    def lookup(data, train, test):
        for row in data:
            if row["training_set"] == train and row["test_set"] == test:
                return row
        return None

    estimators = [
        ("Linguistic\\\\Confidence", "Linguistic Confidence"),
        ("Token\\\\Probability",     "Token Probability"),
        ("Semantic\\\\Uncertainty",  "Semantic Uncertainty"),
    ]
    datasets = ["MMLU", "SQuAD2.0", "TruthfulQA"]

    def build_block(data, metric_label):
        lines = []
        lines.append(rf"\multirow{{9}}{{=}}{{\textbf{{{metric_label}}}}}")

        for ei, (est_display, est_key) in enumerate(estimators):
            lines.append(rf"& \multirow{{3}}{{=}}{{{est_display}}}")

            for ti, train in enumerate(datasets):
                cells = []
                for col in datasets:
                    # if col == train:
                    #     cells.append("-")
                    # else:
                    row = lookup(data, train, col)
                    cells.append(fmt(row.get(est_key), pct=pct) if row else "-")

                cell_str = " & ".join(cells)

                if ti == 0:
                    lines.append(rf"& {train} & {cell_str} \\")
                else:
                    lines.append(rf"& & {train} & {cell_str} \\")

            if ei < len(estimators) - 1:
                lines.append(r"\cmidrule(lr){2-6}")

        return lines

    if pct:
        latex_lines = [
            r"\begin{table}[t]",
            r"\centering",
            r"\caption{In-domain and cross-domain linguistic-space calibration metric percentage changes for both Faithfulness Divergence and generalised ECE. We report percentage change relative to the pre-calibration metrics. Green text indicates calibration improvement (lower error), whereas red text indicates calibration deterioration (higher error). Averaged across all models, token probability and semantic uncertainty exhibits strong transferability for improving faithfulness and calibration, whilst linguistic confidence shows more limited effectiveness.}",
            r"\label{tab:cross-domain-calibration}",
            r"\small",
            r"\begin{tabular}{p{2cm}p{2cm}lccc}",
            r"\toprule",
            r"\textbf{Metric} & \textbf{Estimator} & \textbf{Train/Test} & {MMLU} & SQuAD2.0 & {TruthfulQA} \\",
            r"\midrule",
        ]
    else:
        latex_lines = [
            r"\begin{table}[t]",
            r"\centering",
            r"\caption{In-domain and cross-domain linguistic-space calibration metric changes for both Faithfulness Divergence and generalised ECE. We report the value change relative to the pre-calibration metrics. Green text indicates calibration improvement (lower error), whereas red text indicates calibration deterioration (higher error). Averaged across all models, token probability and semantic uncertainty exhibits strong transferability for improving faithfulness and calibration, whilst linguistic confidence shows more limited effectiveness.}",
            r"\label{tab:cross-domain-calibration}",
            r"\small",
            r"\begin{tabular}{p{2cm}p{2cm}lccc}",
            r"\toprule",
            r"\textbf{Metric} & \textbf{Estimator} & \textbf{Train/Test} & {MMLU} & SQuAD2.0 & {TruthfulQA} \\",
            r"\midrule",
        ]

    latex_lines.extend(build_block(faithfulness_data, "Faithfulness\\\\Divergence\\\\Mean\\\\Reduction"))
    latex_lines.append(r"\midrule")
    latex_lines.extend(build_block(ece_data, "Generalised\\\\ECE\\\\Mean\\\\Reduction"))

    latex_lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ]

    return "\n".join(latex_lines)

print(generate_latex_table(fd_improvement_df.to_dict("records") + in_domain_fd_df.to_dict("records"), 
                           ece_improvement_df.to_dict("records") + in_domain_ece_df.to_dict("records"), 
                           pct=pct))

\begin{table}[t]
\centering
\caption{In-domain and cross-domain linguistic-space calibration metric changes for both Faithfulness Divergence and generalised ECE. We report the value change relative to the pre-calibration metrics. Green text indicates calibration improvement (lower error), whereas red text indicates calibration deterioration (higher error). Averaged across all models, token probability and semantic uncertainty exhibits strong transferability for improving faithfulness and calibration, whilst linguistic confidence shows more limited effectiveness.}
\label{tab:cross-domain-calibration}
\small
\begin{tabular}{p{2cm}p{2cm}lccc}
\toprule
\textbf{Metric} & \textbf{Estimator} & \textbf{Train/Test} & {MMLU} & SQuAD2.0 & {TruthfulQA} \\
\midrule
\multirow{9}{=}{\textbf{Faithfulness\\Divergence\\Mean\\Reduction}}
& \multirow{3}{=}{Linguistic\\Confidence}
& MMLU & \textcolor{green!70!black}{$\Delta$0.1395} & \textcolor{green!70!black}{$\Delta$0.8792} & \textcolor{green!70!black}{$